# Notebook 01: Data Preprocessing and Institutional Aggregation

**1: Project Context**

This notebook serves as the initial stage of the ENAMED 2025 predictive analysis pipeline. Due to Brazilian General Data Protection Law (LGPD) regulations, the microdata prevents direct merging of individual student performance with socioeconomic responses.  

Objective:
Transform student-level microdata into an institutional-level dataset by grouping information by Course Code (CO_CURSO).

**2: Setup and path configuration**

In [19]:
import pandas as pd
import numpy as np
import os

# Paths adjusted for the current 'notebooks/' directory
RAW_DATA_PATH = '../data/raw/'
PROCESSED_DATA_PATH = '../data/processed/'

# UFJF Specific Codes
UFJF_JF_CODE = 13103
UFJF_GV_CODE = 5001167

print("Environment and paths configured.")

Environment and paths configured.


**3: Processing Target Variable (Student Performance - Arq 3)**

The goal here is to calculate the average General Grade (NT_GER) for every medical course in Brazil.

In [20]:
# Loading performance data from data/raw/
# Using ';' as separator and '.' for decimals as per INEP manual
filename = 'microdados_enade_2025_arq3.txt'
df_performance = pd.read_csv(os.path.join(RAW_DATA_PATH, filename), 
                             sep=';', decimal='.', 
                             usecols=['CO_CURSO', 'TP_PR_GER', 'NT_GER'])

# Filter only for students present in the exam (TP_PR_GER == 555)[cite: 1]
df_present = df_performance[df_performance['TP_PR_GER'] == 555].copy()

# Aggregate by Course to calculate the mean grade per institution[cite: 1]
course_performance = df_present.groupby('CO_CURSO')['NT_GER'].mean().reset_index()
course_performance.columns = ['CO_CURSO', 'avg_score_general']

print(f"Aggregation complete. Found {len(course_performance)} unique courses.")

Aggregation complete. Found 350 unique courses.


**4: Final Export (UFJF Identification)**

We tag the UFJF campuses for easier identification in the upcoming Exploratory Data Analysis

In [21]:
# Identify UFJF rows
course_performance['is_ufjf'] = course_performance['CO_CURSO'].isin([UFJF_JF_CODE, UFJF_GV_CODE])

# Save the filtered/aggregated results to data/processed/
output_file = os.path.join(PROCESSED_DATA_PATH, 'enamed_2025_aggregated_performance.csv')
course_performance.to_csv(output_file, index=False)

print(f"Institutional dataset saved to: {output_file}")
# Quick check for UFJF JF presence
print(f"UFJF Juiz de Fora present in results: {UFJF_JF_CODE in course_performance['CO_CURSO'].values}")

Institutional dataset saved to: ../data/processed/enamed_2025_aggregated_performance.csv
UFJF Juiz de Fora present in results: True
